# Phase 3: Embedding Model Fine-Tuning with QLoRA

This notebook fine-tunes the project's exact embedding model (**`BAAI/bge-base-en-v1.5`**) using **4-bit QLoRA (PEFT + BitsAndBytes)** with an **InfoNCE Contrastive / MultipleNegativesRankingLoss** on the enterprise corpus and benchmark.

### Pipeline Context:
$$\text{Query} \rightarrow \mathbf{\text{Fine-Tuned Embeddings}} \rightarrow \text{Phase 3 FAISS Index} \rightarrow \mathbf{\text{Phase 2 FT Reranker}} \rightarrow \text{Baseline LLM} \rightarrow \text{Answer}$$

---

## 1. Environment Check & 2. Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.44.0 peft>=0.12.0 bitsandbytes>=0.43.0 accelerate>=0.33.0 datasets>=2.20.0 rank-bm25>=0.2.2 sentence-transformers>=3.0.0 safetensors

import os
import sys
import json
import random
import shutil
import time
import math
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

import transformers
import peft
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from rank_bm25 import BM25Okapi

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Hyperparameters & Configuration

In [ ]:
@dataclass
class HyperParameters:
    base_model_name: str = "BAAI/bge-base-en-v1.5"
    embedding_dim: int = 768
    query_prefix: str = "Represent this sentence for searching relevant passages: "
    max_query_length: int = 128
    max_doc_length: int = 400
    
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: Tuple[str, ...] = ("query", "key", "value", "dense")
    
    temperature: float = 0.05
    num_train_epochs: int = 5
    learning_rate: float = 2e-4
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    seed: int = 42
    num_hard_negatives: int = 2
    
    train_ratio: float = 0.80
    val_ratio: float = 0.10
    test_ratio: float = 0.10
    
    questions_file: str = "questions_visible.jsonl"
    chunks_file: str = "chunks.jsonl"
    package_dir: str = "/kaggle/working/embedding_phase3"

CONFIG = HyperParameters()

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG.seed)
print("Config loaded successfully:")
print(json.dumps(CONFIG.__dict__, indent=2))

## 4. Load Dataset & 5. Mine Hard Negatives for Contrastive Training

In [ ]:
def locate_file(filename: str) -> str:
    search_paths = [
        filename,
        os.path.join(".", filename),
        os.path.join("..", filename),
        os.path.join("/kaggle/input", filename),
        os.path.join("/kaggle/input/enterprise-rag-dataset", filename),
        os.path.join("/kaggle/working", filename),
    ]
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    for p in search_paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not locate '{filename}'")

q_path = locate_file(CONFIG.questions_file)
c_path = locate_file(CONFIG.chunks_file)

questions_data = [json.loads(line) for line in open(q_path, "r", encoding="utf-8") if line.strip()]
chunks_dict = {json.loads(line)["chunk_id"]: json.loads(line) for line in open(c_path, "r", encoding="utf-8") if line.strip()}

chunk_id_list = list(chunks_dict.keys())
corpus_texts = [chunks_dict[cid]["text"] for cid in chunk_id_list]
tokenized_corpus = [t.lower().split() for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

triplet_records = []
for item in questions_data:
    gold_cids = set(item.get("relevant_chunk_ids", []))
    pos_texts = [chunks_dict[cid]["text"] for cid in gold_cids if cid in chunks_dict]
    if not pos_texts:
        continue
        
    query_tokens = item["question"].lower().split()
    top_indices = bm25.get_top_n(query_tokens, list(range(len(chunk_id_list))), n=35)
    neg_texts = [chunks_dict[chunk_id_list[idx]]["text"] for idx in top_indices if chunk_id_list[idx] not in gold_cids][:CONFIG.num_hard_negatives]
    
    for pos in pos_texts:
        triplet_records.append({
            "id": item["id"],
            "query": CONFIG.query_prefix + item["question"],
            "positive": pos,
            "negatives": neg_texts,
            "raw_query": item["question"],
            "gold_cids": list(gold_cids),
        })

print(f"Total Questions: {len(questions_data)} | Total Chunks: {len(chunks_dict):,}")
print(f"Constructed {len(triplet_records)} contrastive training triplets.")

## 6. Query-Level Train / Validation / Test Split

In [ ]:
unique_qids = list(set(r["id"] for r in triplet_records))
random.seed(CONFIG.seed)
random.shuffle(unique_qids)

n_total = len(unique_qids)
n_train = int(n_total * CONFIG.train_ratio)
n_val = int(n_total * CONFIG.val_ratio)

train_qids = set(unique_qids[:n_train])
val_qids   = set(unique_qids[n_train:n_train + n_val])
test_qids  = set(unique_qids[n_train + n_val:])

train_triplets = [r for r in triplet_records if r["id"] in train_qids]
val_triplets   = [r for r in triplet_records if r["id"] in val_qids]
test_triplets  = [r for r in triplet_records if r["id"] in test_qids]

print(f"Split Sizes -> Train: {len(train_qids)} queries ({len(train_triplets)} triplets)")
print(f"               Val:   {len(val_qids)} queries ({len(val_triplets)} triplets)")
print(f"               Test:  {len(test_qids)} queries ({len(test_triplets)} triplets)")

## 7 & 8. Load Base Embedding Model & Apply QLoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG.base_model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

print(f"Loading base model '{CONFIG.base_model_name}' in 4-bit precision...")
base_transformer = AutoModel.from_pretrained(
    CONFIG.base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
base_transformer = prepare_model_for_kbit_training(base_transformer)

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=CONFIG.lora_r,
    lora_alpha=CONFIG.lora_alpha,
    lora_dropout=CONFIG.lora_dropout,
    target_modules=list(CONFIG.target_modules),
    bias="none",
)

peft_embedding_model = get_peft_model(base_transformer, peft_config)
peft_embedding_model.print_trainable_parameters()

## 9. Contrastive Embedding Training (with Epoch Validation Loss)

In [ ]:
class EmbeddingTripletDataset(Dataset):
    def __init__(self, triplets):
        self.triplets = triplets
    def __len__(self):
        return len(self.triplets)
    def __getitem__(self, idx):
        item = self.triplets[idx]
        neg = item["negatives"][0] if item["negatives"] else item["positive"]
        return {"query": item["query"], "positive": item["positive"], "negative": neg}

train_loader = DataLoader(EmbeddingTripletDataset(train_triplets), batch_size=CONFIG.batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(EmbeddingTripletDataset(val_triplets), batch_size=CONFIG.batch_size, shuffle=False)

def encode_texts(model, texts, max_len, device):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    outputs = model(**inputs)
    cls_rep = outputs.last_hidden_state[:, 0]
    return F.normalize(cls_rep, p=2, dim=1)

class MultipleNegativesRankingLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature
        self.ce = nn.CrossEntropyLoss()
    def forward(self, q_emb, p_emb, n_emb):
        sim_pos = torch.matmul(q_emb, p_emb.T) / self.temperature
        sim_neg = torch.matmul(q_emb, n_emb.T) / self.temperature
        scores = torch.cat([sim_pos, sim_neg], dim=1)
        labels = torch.arange(q_emb.size(0), device=q_emb.device)
        return self.ce(scores, labels)

loss_fn = MultipleNegativesRankingLoss(temperature=CONFIG.temperature)
optimizer = torch.optim.AdamW(peft_embedding_model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
device = next(peft_embedding_model.parameters()).device

print(f"Training on {device} for {CONFIG.num_train_epochs} epochs...")
t0_train = time.time()

for epoch in range(1, CONFIG.num_train_epochs + 1):
    peft_embedding_model.train()
    total_train_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, 1):
        q_emb = encode_texts(peft_embedding_model, batch["query"], CONFIG.max_query_length, device)
        p_emb = encode_texts(peft_embedding_model, batch["positive"], CONFIG.max_doc_length, device)
        n_emb = encode_texts(peft_embedding_model, batch["negative"], CONFIG.max_doc_length, device)
        loss = loss_fn(q_emb, p_emb, n_emb) / CONFIG.gradient_accumulation_steps
        loss.backward()
        total_train_loss += loss.item() * CONFIG.gradient_accumulation_steps
        if step % CONFIG.gradient_accumulation_steps == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Compute Validation Loss
    peft_embedding_model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            q_emb = encode_texts(peft_embedding_model, batch["query"], CONFIG.max_query_length, device)
            p_emb = encode_texts(peft_embedding_model, batch["positive"], CONFIG.max_doc_length, device)
            n_emb = encode_texts(peft_embedding_model, batch["negative"], CONFIG.max_doc_length, device)
            v_loss = loss_fn(q_emb, p_emb, n_emb)
            total_val_loss += v_loss.item()
            
    avg_val_loss = total_val_loss / max(len(val_loader), 1)
    print(f"Epoch {epoch}/{CONFIG.num_train_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

train_time_min = (time.time() - t0_train) / 60.0
print(f"Fine-Tuning finished in {train_time_min:.2f} mins.")

## 10 & 11. Full-Corpus Retrieval Evaluation: Base vs Fine-Tuned Embedding

In [ ]:
# Search against ALL 10,549 chunks so gold chunks are 100% guaranteed to be present
eval_chunks = [chunks_dict[cid] for cid in chunk_id_list]
corpus_cids = [c["chunk_id"] for c in eval_chunks]
print(f"Total Corpus Search Pool: {len(eval_chunks):,} chunks")

def compute_corpus_embeddings(model_to_use, chunks, batch_size=64):
    model_to_use.eval()
    all_embeddings = []
    texts = [c["text"] for c in chunks]
    device = next(model_to_use.parameters()).device
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            emb = encode_texts(model_to_use, texts[i:i+batch_size], CONFIG.max_doc_length, device)
            all_embeddings.append(emb.cpu().numpy().astype(np.float32))
    return np.vstack(all_embeddings)

print("Encoding full corpus with Fine-Tuned Embedding Model...")
ft_corpus_emb = compute_corpus_embeddings(peft_embedding_model, eval_chunks, batch_size=64)

print("Encoding full corpus with Base Pretrained Embedding Model...")
base_eval_model = AutoModel.from_pretrained(CONFIG.base_model_name, quantization_config=bnb_config, device_map="auto")
base_corpus_emb = compute_corpus_embeddings(base_eval_model, eval_chunks, batch_size=64)

def evaluate_retrieval(model_to_use, corpus_emb, test_items, k_vals=[1, 5, 10]):
    model_to_use.eval()
    device = next(model_to_use.parameters()).device
    r1, r5, r10, mrr, ndcg = [], [], [], [], []
    
    with torch.no_grad():
        for item in test_items:
            q_text = CONFIG.query_prefix + item["raw_query"]
            q_emb = encode_texts(model_to_use, [q_text], CONFIG.max_query_length, device).cpu().numpy()
            
            scores = np.dot(corpus_emb, q_emb.T).flatten()
            ranked_indices = np.argsort(scores)[::-1][:max(k_vals)]
            ranked_cids = [corpus_cids[idx] for idx in ranked_indices]
            gold_set = set(item["gold_cids"])
            
            r1.append(1.0 if any(cid in gold_set for cid in ranked_cids[:1]) else 0.0)
            r5.append(1.0 if any(cid in gold_set for cid in ranked_cids[:5]) else 0.0)
            r10.append(1.0 if any(cid in gold_set for cid in ranked_cids[:10]) else 0.0)
            
            first_hit = next((rank + 1 for rank, cid in enumerate(ranked_cids[:10]) if cid in gold_set), 0)
            mrr.append(1.0 / first_hit if first_hit > 0 else 0.0)
            
            dcg = sum((1.0 if cid in gold_set else 0.0) / math.log2(r + 2) for r, cid in enumerate(ranked_cids[:10]))
            ndcg.append(dcg)
            
    return {
        "recall_at_1": float(np.mean(r1)),
        "recall_at_5": float(np.mean(r5)),
        "recall_at_10": float(np.mean(r10)),
        "mrr_at_10": float(np.mean(mrr)),
        "ndcg_at_10": float(np.mean(ndcg)),
    }

print("\nEvaluating Dense Retrieval Metrics on Test Set...")
base_metrics = evaluate_retrieval(base_eval_model, base_corpus_emb, test_triplets)
ft_metrics   = evaluate_retrieval(peft_embedding_model, ft_corpus_emb, test_triplets)

del base_eval_model
torch.cuda.empty_cache()

print("\n" + "=" * 75)
print("PHASE 3 EMBEDDING EVALUATION COMPARISON (DENSE RETRIEVAL TEST SET)")
print("=" * 75)
print(f"{'Metric':<18} | {'Base Embedding':<18} | {'Fine-Tuned (QLoRA)':<22} | {'Delta':<10}")
print("-" * 75)
for k in ["recall_at_1", "recall_at_5", "recall_at_10", "mrr_at_10", "ndcg_at_10"]:
    b_val = base_metrics[k]
    f_val = ft_metrics[k]
    delta = f_val - b_val
    sign = "+" if delta >= 0 else ""
    print(f"{k:<18} | {b_val:<18.4f} | {f_val:<22.4f} | {sign}{delta:.4f}")
print("=" * 75)

## 14 & 16. Save Artifacts & Package for Local Integration

In [ ]:
package_dir = Path(CONFIG.package_dir)
adapter_dir = package_dir / "adapter"
tokenizer_dir = package_dir / "tokenizer"

if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True, exist_ok=True)
adapter_dir.mkdir(parents=True, exist_ok=True)
tokenizer_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapter to {adapter_dir}...")
peft_embedding_model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(tokenizer_dir))

eval_summary = {
    "base_model": CONFIG.base_model_name,
    "fine_tuning_method": "QLoRA (4-bit NF4 + PEFT LoRA)",
    "training_time_minutes": round(train_time_min, 2),
    "metrics_comparison": {
        "base_embedding": base_metrics,
        "fine_tuned_embedding_qlora": ft_metrics,
    }
}
with open(package_dir / "evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_summary, f, indent=2)

with open(package_dir / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG.__dict__, f, indent=2)

shutil.make_archive("/kaggle/working/embedding_phase3", "zip", package_dir)
print("\n[SUCCESS] Package archive ready at: /kaggle/working/embedding_phase3.zip")